# Descarga de cuerpos de noticias del WSJ

Entrada: `data/wsj2014_2020.csv`. Se conserva cada fila original, sin filtro temático.
El trabajo avanza por **un día → una semana → un mes → un año**, con la misma fecha
inicial y una carpeta compartida de checkpoints para reutilizar lo ya procesado.

El CSV inspeccionado contiene **110.156 filas**, desde **2014-07-01 hasta 2020-12-31**;
el nombre del archivo no implica que incluya todo 2014. Sus fechas usan **`%d/%m/%Y`**.
Hay URLs duplicadas, vacías y externas al WSJ: se reportan, sin corregirlas silenciosamente.

**Resultado esperado:** cuerpos recuperables y evidencia de los faltantes. Un HTTP 200,
un `articleBody` o un texto largo **no garantizan** que el artículo esté completo.
`candidate_unverified` identifica texto para revisión; `full_text_verified` permanece
en `False`. Este notebook es una herramienta de adquisición, no una muestra analítica validada.

## 0. Entorno y vías de recuperación

Selecciona el kernel Python del `.venv` del proyecto. Si falta el entorno, ejecuta
`uv sync` en la raíz; para el navegador opcional, `uv sync --extra browser`.
No se instalan paquetes automáticamente desde el notebook.

| Vía implementada | Mecanismo y límite |
|---|---|
| HTML directo | HTTPS de la URL original, redirecciones auditadas y consulta de robots por dominio. |
| Dominio histórico | Prueba `www.wsj.com` cuando `online.wsj.com` responde 404/410. |
| Extracción local | JSON-LD `articleBody`, contenedores del cuerpo y trafilatura sobre la misma respuesta. |
| Navegador opcional | Edge/Chrome/Chromium convencional para HTML accesible que necesita JavaScript. |
| Wayback | API pública de disponibilidad y captura de la URL exacta, si existe y es accesible. |
| HTML autorizado local | Importa un HTML obtenido previamente, identificado por `article_id`. |

Una denegación, paywall o verificación se registra como tal. No se usan proxies,
suplantación de buscadores, resolvedores de CAPTCHA ni endpoints privados. Robots
restrictivo y robots no disponible se distinguen; ambos detienen esa ruta en esta versión.
Wayback es otra plataforma de adquisición y su timestamp **no sustituye** la publicación.

Para faltantes persistentes, Factiva/Dow Jones u otros archivos institucionales son rutas
por evaluar con el acceso y exportaciones disponibles. No se inventa una API de texto
completo ni se asume una licencia. Common Crawl también requiere comprobar capturas y
derechos; su descarga WARC no está implementada en este notebook.

Referencias técnicas: [trafilatura](https://trafilatura.readthedocs.io/en/latest/quickstart.html),
[Playwright](https://playwright.dev/python/docs/network),
[API de Wayback](https://archive.org/help/wayback_api.php),
[índice Common Crawl](https://index.commoncrawl.org/).
Los textos se guardan bajo `data/`, ignorado por Git; su acceso no concede derechos de redistribución.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file()
             and (p / "src" / "wsj_download.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abre este notebook dentro del repositorio DCCS-Gold-Thesis")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsj_download import load_input, period_plan

required = ["requests", "bs4", "trafilatura", "truststore"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Faltan dependencias: {missing}. Ejecuta uv sync y selecciona el kernel del proyecto")
print("Raíz:", ROOT)
print("Python:", sys.executable)

## 1. Leer y auditar el CSV, sin descargar

La fecha del CSV se trata como fecha de calendario sin zona horaria conocida.
Las ventanas incluyen el inicio y excluyen el fin. Un mes y un año usan desplazamientos
de calendario, no 30 y 365 días fijos. La ventana anual puede cruzar dos años.
El conteo de URLs únicas de descarga agrupa equivalentes `online`/`www`; las filas originales se conservan.

In [ ]:
CSV_PATH = ROOT / "data" / "wsj2014_2020.csv"
OUTPUT_DIR = ROOT / "data" / "wsj_download"
news = load_input(CSV_PATH)
START_DATE = news["csv_date"].min().date().isoformat()  # Cambiar para otro bloque, p. ej. "2017-01-01".

display(pd.Series({
    "filas_csv": len(news),
    "urls_originales_distintas": news["url"].nunique(),
    "identificadores_de_descarga": news["article_id"].nunique(),
    "fecha_mínima": str(news["csv_date"].min().date()),
    "fecha_máxima": str(news["csv_date"].max().date()),
    "filas_con_incidencias": int((news["input_status"] != "ok").sum()),
}))
display(news["input_status"].value_counts(dropna=False).rename("filas"))
display(news.loc[news["input_status"] != "ok",
                 ["source_row", "date", "url", "input_status"]].head(20))
display(period_plan(news, START_DATE))

## 2. Configurar la descarga

Al ejecutar todas las celdas solo se inicia la etapa de un día. Para ampliar, activa
la etapa correspondiente después de revisar sus resultados. Cada etapa procesa todas
las URLs de su ventana salvo que se indique `MAX_NEW_ARTICLES` o se agote el presupuesto HTTP.
Estos límites se expresan en **intentos**, nunca en éxitos garantizados.

Para el inicio observado, las etapas contienen 50, 321, 1.451 y 16.969 filas,
respectivamente. Se usa una pausa de 3 segundos por host, ampliable por robots y
`Retry-After`; el año puede tardar muchas horas. Las descargas son secuenciales.

In [ ]:
RUN_DAY = True
RUN_WEEK = False
RUN_MONTH = False
RUN_YEAR = False

USE_WAYBACK = True
USE_BROWSER = False                 # Opcional; requiere Playwright y navegador instalado.
BROWSER_CHANNEL = "msedge"         # "chrome" o "chromium" si están disponibles.
LOCAL_HTML_DIR = None               # Ej.: ROOT / "data" / "wsj_authorized_html"
RETRY_FAILED = False                # True vuelve a probar fallos; conserva evidencia previa.
MAX_NEW_ARTICLES = None             # Ej.: 3 para una primera prueba pequeña.
DELAY_SECONDS = 3.0
TIMEOUT_SECONDS = 25.0
RETRIES = 2
MIN_WORDS_CANDIDATE = 80            # Indicador técnico; los textos más cortos también se guardan.
REQUEST_BUDGET = {"day": 500, "week": 3000, "month": 12000, "year": 150000}

# Se mantiene una carpeta común entre etapas. Cambios en CSV, extractor o vías de
# acceso generan otro perfil de checkpoint y no reutilizan resultados incompatibles.
results = {}

El motor se ejecuta en un proceso Python separado, usando el mismo intérprete que el
kernel. Así el navegador síncrono de Playwright no entra en conflicto con el bucle
asíncrono de Jupyter. El log aparece a medida que termina cada URL.

In [ ]:
def download_stage(stage, enabled):
    if not enabled:
        print(f"Etapa {stage} desactivada; cambia su RUN_* para ejecutarla.")
        return None
    command = [sys.executable, "-u", str(ROOT / "src" / "wsj_download.py"),
               "--csv", str(CSV_PATH), "--output", str(OUTPUT_DIR),
               "--start", START_DATE, "--stage", stage,
               "--max-requests", str(REQUEST_BUDGET[stage]),
               "--delay", str(DELAY_SECONDS), "--timeout", str(TIMEOUT_SECONDS),
               "--retries", str(RETRIES), "--min-words", str(MIN_WORDS_CANDIDATE)]
    if not USE_WAYBACK:
        command.append("--no-wayback")
    if USE_BROWSER:
        command.extend(["--browser", "--browser-channel", BROWSER_CHANNEL])
    if LOCAL_HTML_DIR is not None:
        command.extend(["--local-html", str(LOCAL_HTML_DIR)])
    if RETRY_FAILED:
        command.append("--retry-failed")
    if MAX_NEW_ARTICLES is not None:
        command.extend(["--max-articles", str(MAX_NEW_ARTICLES)])
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                               errors="replace", env={**__import__("os").environ, "PYTHONIOENCODING": "utf-8"})
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        print("Interrumpido. Los documentos y checkpoints ya escritos se conservan.")
        print("Si quedó .download.lock, elimínalo solo tras comprobar que el proceso terminó.")
        return None
    finally:
        process.stdout.close()
    if return_code != 0:
        raise RuntimeError(f"El proceso terminó con código {return_code}; revisa el log anterior")
    summary = json.loads((OUTPUT_DIR / "latest_run.json").read_text(encoding="utf-8"))
    results[stage] = summary
    return summary


def inspect_stage(summary):
    if summary is None:
        return None
    run_dir = Path(summary["run_dir"])
    rows = pd.read_csv(run_dir / "all_rows.csv", keep_default_na=False)
    display(pd.Series({key: summary[key] for key in ["csv_rows", "unique_articles", "cached_articles",
        "requests_sent", "candidate_articles", "pending_articles", "request_budget_exhausted"]}))
    display(rows["status"].value_counts().rename("filas"))
    display(rows.groupby(["acquisition_platform", "status"], dropna=False).size().rename("filas"))
    return rows

## 3. Un día de noticias

Empieza aquí. Revisa errores, procedencia, titulares y cuerpos antes de ampliar.
Reejecutar con la misma configuración reutiliza tanto éxitos como fallos ya registrados;
para volver a probar fallos, cambia `RETRY_FAILED=True` y ejecuta de nuevo la celda de configuración.

In [ ]:
day_summary = download_stage("day", RUN_DAY)
day_rows = inspect_stage(day_summary)

In [ ]:
# Vista breve de los textos recuperados. El cuerpo completo queda en documents/ y en los CSV.
if day_rows is not None:
    recovered = day_rows.loc[day_rows["text"].str.len() > 0].copy()
    recovered["body_preview"] = recovered["text"].str.slice(0, 1800)
    display(recovered[["headlines", "url", "status", "word_count", "acquisition_platform", "body_preview"]].head(5))

## 4. Una semana de noticias

La ventana cubre siete días desde `START_DATE`. El día anterior queda dentro y se reutiliza.
Si el presupuesto se agota, quedan filas `pending`; repetir la etapa continúa lo pendiente.

In [ ]:
week_summary = download_stage("week", RUN_WEEK)
week_rows = inspect_stage(week_summary)

## 5. Un mes de noticias

La ventana termina en la misma fecha del mes siguiente, o en el último día válido.
Se conservan también noticias de deportes, opinión y otras secciones presentes en el CSV.

In [ ]:
month_summary = download_stage("month", RUN_MONTH)
month_rows = inspect_stage(month_summary)

## 6. Un año de noticias

Se amplía hasta la misma fecha del año siguiente. Para un año de calendario, establece
`START_DATE="2017-01-01"`, por ejemplo. No se inicia automáticamente al ejecutar todo.

In [ ]:
year_summary = download_stage("year", RUN_YEAR)
year_rows = inspect_stage(year_summary)

## 7. Revisar cobertura, fallos y resultados

Las tasas usan el CSV como denominador. No estiman todo lo publicado por WSJ. `pending`
significa aún no resuelto en esta ejecución; los demás estados describen resultados observados.
Las fechas inválidas se muestran en la auditoría global y no se asignan a ventanas temporales.
Las URLs fuera del alcance quedan registradas sin petición; no se reinterpretan como noticias WSJ.

In [ ]:
if results:
    comparison = pd.DataFrame([{ "stage": stage, **{k: s[k] for k in [
        "csv_rows", "unique_articles", "cached_articles", "requests_sent", "candidate_articles",
        "candidate_fraction_of_period", "pending_articles", "full_text_verified_count"]}}
        for stage, s in results.items()])
    display(comparison)
    latest = next(reversed(results.values()))
    latest_rows = pd.read_csv(Path(latest["run_dir"]) / "all_rows.csv", keep_default_na=False)
    display(latest_rows.assign(month=pd.to_datetime(latest_rows["date"], format="%d/%m/%Y").dt.strftime("%Y-%m"))
            .groupby(["month", "status"]).size().unstack(fill_value=0))
    # Para inspeccionar métodos y causas de cada ruta, abrir los JSON de documents/.
    http_path = Path(latest["run_dir"]) / "http.jsonl"
    attempts = [json.loads(line) for line in http_path.read_text(encoding="utf-8").splitlines() if line]
    if attempts:
        display(pd.DataFrame(attempts).groupby(["stage", "status"], dropna=False).size().rename("eventos"))
else:
    print("Todavía no hay etapas ejecutadas en esta sesión.")

## 8. Salidas, reanudación y HTML local

```text
data/wsj_download/
  raw/<sha256>.bin                   Respuestas originales, identificadas por hash
  documents/<article_id>/<run>.json   Texto, metadatos y resultado de cada vía
  checkpoint.json                    Último resultado por URL y perfil; esperas por host
  latest_run.json                    Resumen de la última invocación
  runs/<run_id>/
    manifest.json                    CSV/código SHA-256, Git, entorno, versiones y configuración
    http.jsonl                       Peticiones y decisiones de robots diferenciadas
    attempts.jsonl                   Resultados por identificador de descarga
    all_rows.csv                     Todas las filas de la ventana, incluidos fallos y pendientes
    articles_candidates.csv           Cuerpos candidatos; completitud pendiente de revisión
    summary.json                     Denominadores, tasas, reanudación y límites alcanzados
```

Los originales y documentos anteriores se conservan. `checkpoint.json` y `latest_run.json`
son índices actualizables; cada corrida genera un directorio nuevo. No ejecutar dos procesos
sobre la misma salida. Si se corta abruptamente puede quedar `.download.lock`: comprobar que
el proceso terminó antes de retirarlo. Repetir con la misma carpeta reanuda desde el último
documento terminado. La URL interrumpida puede necesitar repetirse. Las esperas de más de
60 segundos quedan diferidas y persistidas, sin mantener una celda dormida durante horas.

Para HTML obtenido por una vía autorizada, guarda cada archivo como
`data/wsj_authorized_html/<article_id>.html` y configura `LOCAL_HTML_DIR`.
Los identificadores se consultan en `news[["article_id", "url", "headlines"]]`.
El HTML local debe contener el cuerpo y, cuando existan, metadatos de identidad;
un avance sigue siendo un avance. Si cambias archivos locales ya procesados, usa una
**carpeta OUTPUT_DIR nueva** para mantener la procedencia y volver a extraer.

Para otra versión del extractor, otro CSV o cambios en vías de acceso se calcula un perfil
distinto. La misma URL puede tener varias capturas: no se confunden con artículos independientes.
El hash de texto permite detectar coincidencias exactas sin borrar originales o republicaciones.

Una revisión humana de completitud debe guardarse aparte, vinculada al `article_id`, `run_id`
y hash del texto; no editar manualmente las respuestas ni los resultados originales.

Las pruebas del motor se ejecutan sin red con:
`uv run --offline --no-sync python -m unittest discover -s tests -p test_wsj_download.py -v`.

## Verificación al crear el notebook — 2026-11-18

- **31 pruebas locales** aprobadas con transporte simulado y HTML sintético.
- Se ejecutaron todas las celdas de código, incluidas las cuatro etapas, con un HTML
  sintético local: cero solicitudes remotas y reutilización del checkpoint entre etapas.
- Piloto real acotado sobre tres URLs del primer día: **10 solicitudes HTTP**, incluidas
  consultas de robots y Wayback. La ruta directa quedó detenida por robots; Wayback
  devolvió tres capturas. Tras corregir la interpretación de sus URLs canónicas se
  reprocesaron esos mismos archivos **sin solicitudes adicionales**.
- Se extrajeron **dos cuerpos candidatos de 215 y 188 palabras**; la tercera captura
  declaró `article.access=paid` y quedó como `restricted_content`. No se certificó la
  completitud ni se ejecutaron la semana, el mes o el año contra internet.

Evidencia local: `data/diagnostics/wsj_remote_20261118/reextracted/summary.json`;
los documentos reprocesados están en la misma carpeta. Las respuestas y registros
de adquisición permanecen en `data/diagnostics/wsj_remote_20261118/`.
La prueba de celdas está en `data/diagnostics/wsj_validation/notebook_smoke/execution.log`.
Los resultados son de ingeniería y no establecen cobertura para todo el CSV.